# 03 · Product recommendations

`nodeSimilarity` over the co-purchase graph, using an Aura Graph Analytics
session, written back to OneLake.

Products are similar when they keep the same company — bought alongside the same
other products. That differs from direct co-purchase counting, which just returns
whatever is popular.

Four settings do the work:

| | |
|---|---|
| undirected projection | directed without it, and similarity becomes a function of `productId` |
| `MIN_PRODUCT_ORDERS` | excludes rarely-bought products, whose few near-identical baskets produce inflated scores |
| `MIN_WEIGHT` | drops pairs that co-occurred too few times to mean anything |
| `degreeCutoff` | excludes products with small neighbourhoods, where Jaccard is unstable |

## Setup

In [22]:
%pip install graphdatascience

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 36, Finished, Available, Finished, True)


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [23]:
from graphdatascience.session import (
        GdsSessions,
        AuraAPICredentials,
        DbmsConnectionInfo,
        AlgorithmCategory,
    )

from datetime import timedelta
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 400)
print("imports OK")

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 38, Finished, Available, Finished, False)

imports OK


In [24]:
with open("/lakehouse/default/Files/fabric_keys.json") as f:
    secrets = json.load(f)

NEO4J_URI      = secrets["NEO4J_URI"]
NEO4J_USERNAME = secrets["NEO4J_USERNAME"]
NEO4J_PASSWORD = secrets["NEO4J_PASSWORD"]
CLIENT_ID      = secrets["CLIENT_ID"]
CLIENT_SECRET  = secrets["CLIENT_SECRET"]
TENANT_ID      = secrets["TENANT_ID"]

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 39, Finished, Available, Finished, False)

In [25]:
PRODUCT_KEY  = "productId"
PRODUCT_NAME = "productName"

MIN_PRODUCT_ORDERS = 100    # product must appear in this many orders
MIN_WEIGHT         = 15     # pair must co-occur this many times
DEGREE_CUTOFF      = 20     # min neighbours to be scored
SIMILARITY_CUTOFF  = 0.05
TOP_K              = 10

# Ask for undirected at projection time instead of mutating a second copy.
REL_TYPE = "CO_PURCHASED_WITH"

OUTPUT_TABLE = "scored_results"
SESSION_NAME = "retail-recommendations"

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 40, Finished, Available, Finished, False)

In [26]:
sessions = GdsSessions(
    api_credentials=AuraAPICredentials(CLIENT_ID, CLIENT_SECRET, TENANT_ID)
)

# SIMILARITY is absent on some client versions.
_cats = [getattr(AlgorithmCategory, n)
         for n in ("SIMILARITY", "NODE_EMBEDDING", "CENTRALITY")
         if hasattr(AlgorithmCategory, n)]

memory = sessions.estimate(
    node_count=50_000,
    relationship_count=12_000_000,
    algorithm_categories=_cats,
)

gds = sessions.get_or_create(
    SESSION_NAME,
    memory=memory,
    db_connection=DbmsConnectionInfo(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD),
    ttl=timedelta(hours=2),
)
print("GDS session ready:", gds.version())

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 41, Finished, Available, Finished, False)

GDS session ready: 2026.8.0


## Step 1 · Direct co-purchase counts

The naive answer, for comparison. Every row will be produce, because popular
products co-occur with everything.

In [27]:
q = f"""
MATCH (p1:Product)<-[:CONTAINS]-(:Order)-[:CONTAINS]->(p2:Product)
WHERE p1.{PRODUCT_KEY} < p2.{PRODUCT_KEY}
WITH p1.{PRODUCT_KEY} AS a, p2.{PRODUCT_KEY} AS b, count(*) AS timesTogether
ORDER BY timesTogether DESC
LIMIT 10
MATCH (x:Product), (y:Product)
WHERE x.{PRODUCT_KEY} = a AND y.{PRODUCT_KEY} = b
RETURN x.{PRODUCT_NAME} AS productA, y.{PRODUCT_NAME} AS productB, timesTogether
"""
gds.run_cypher(q)

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 42, Finished, Available, Finished, False)

,productA,productB,timesTogether
0,Bag of Organic Bananas,Organic Hass Avocado,10043
1,Bag of Organic Bananas,Organic Strawberries,9872
2,Organic Strawberries,Banana,8541
3,Banana,Organic Avocado,8182
4,Organic Baby Spinach,Banana,7856
5,Bag of Organic Bananas,Organic Baby Spinach,7803
6,Organic Strawberries,Organic Hass Avocado,6702
7,Strawberries,Banana,6583
8,Bag of Organic Bananas,Organic Raspberries,6568
9,Banana,Large Lemon,6225


## Step 2 · Product popularity

Stored as a property so the projection can filter on it. The projection
aggregates every co-occurring pair before applying `MIN_WEIGHT`, so filtering
products is the only way to keep it inside AuraDB's transaction memory limit.

In [28]:
r = gds.run_cypher("""
MATCH (p:Product)<-[:CONTAINS]-(:Order)
WITH p, count(*) AS n SET p.orderCount = n
RETURN count(*) AS products, max(n) AS most, round(avg(n)) AS mean
""")
print(r.to_string(index=False))

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 43, Finished, Available, Finished, False)

 products  most  mean
    44536 72846 112.0


In [29]:
rows = []
for thr in (0, 50, 100, 500, 1000, 2000):
    q = f"""
    MATCH (o:Order)-[:CONTAINS]->(p:Product)
    WHERE p.orderCount >= {thr}
    WITH o, count(p) AS k
    RETURN sum(k * (k - 1) / 2) AS pair_rows
    """
    pr = int(gds.run_cypher(q)["pair_rows"].iloc[0])
    n_prod = int(gds.run_cypher(
        f"MATCH (p:Product) WHERE p.orderCount >= {thr} RETURN count(p) AS n"
    )["n"].iloc[0])
    rows.append({"MIN_PRODUCT_ORDERS": thr, "products": n_prod,
                 "pair_rows": pr, "est_GB": round(pr * 300 / 1e9, 2)})

print(pd.DataFrame(rows).to_string(index=False))
print(f"\nCONFIG has MIN_PRODUCT_ORDERS = {MIN_PRODUCT_ORDERS}")
print("Keep est_GB under ~3 - the transaction limit is 3.5 GiB.")

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 44, Finished, Available, Finished, False)

 MIN_PRODUCT_ORDERS  products  pair_rows  est_GB
                  0     44536   36667471   11.00
                 50     11089   31526190    9.46
                100      7007   27981237    8.39
                500      1755   15933526    4.78
               1000       815   10466536    3.14
               2000       341    5924088    1.78

CONFIG has MIN_PRODUCT_ORDERS = 100
Keep est_GB under ~3 - the transaction limit is 3.5 GiB.


## Step 3 · Weight distribution

`MIN_WEIGHT` drops pairs that co-occurred too rarely to be evidence.

In [30]:
q = f"""
MATCH (o:Order)-[:CONTAINS]->(p1:Product), (o)-[:CONTAINS]->(p2:Product)
WHERE p1.{PRODUCT_KEY} < p2.{PRODUCT_KEY}
  AND p1.orderCount >= {MIN_PRODUCT_ORDERS}
  AND p2.orderCount >= {MIN_PRODUCT_ORDERS}
WITH p1, p2, count(*) AS w
WITH CASE WHEN w >= 100 THEN '100+'
          WHEN w >=  20 THEN '20-99'
          WHEN w >=  10 THEN '10-19'
          WHEN w >=   5 THEN '5-9'
          ELSE toString(w) END AS bucket,
     count(*) AS pairs
RETURN bucket, pairs ORDER BY pairs DESC
"""
dist = gds.run_cypher(q)
dist["pct"] = (100 * dist["pairs"] / dist["pairs"].sum()).round(2)
print(dist.to_string(index=False))

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 45, Finished, Available, Finished, False)

bucket   pairs   pct
     1 2619729 46.75
     2 1011273 18.05
   5-9  632302 11.28
     3  525237  9.37
     4  319112  5.69
 10-19  291258  5.20
 20-99  182476  3.26
  100+   22617  0.40


## Step 4 · Project

The `p1.productId < p2.productId` inequality prevents duplicate edges but leaves
the graph directed — a product's neighbourhood becomes only the products with a
higher id, and similarity ends up a function of the id rather than the basket.

`undirected_relationship_types` is a keyword argument on `gds.graph.project()`,
not a key in the Cypher `dataConfig` map. GDS builds one undirected type
directly, so there is no second copy in memory.

In [31]:
if gds.graph.exists("copurchase")["exists"]:
    gds.graph.drop("copurchase")

query = f"""
CALL {{
    MATCH (o:Order)-[:CONTAINS]->(p1:Product),
          (o)-[:CONTAINS]->(p2:Product)
    WHERE p1.{PRODUCT_KEY} < p2.{PRODUCT_KEY}
      AND p1.orderCount >= {MIN_PRODUCT_ORDERS}
      AND p2.orderCount >= {MIN_PRODUCT_ORDERS}
    WITH p1, p2, toFloat(count(*)) AS weight
    WHERE weight >= {MIN_WEIGHT}
    RETURN p1 AS source, p2 AS target, weight
}}
RETURN gds.graph.project.remote(
    source,
    target,
    {{
        sourceNodeLabels: labels(source),
        targetNodeLabels: labels(target),
        sourceNodeProperties: source {{ .{PRODUCT_KEY} }},
        targetNodeProperties: target {{ .{PRODUCT_KEY} }},
        relationshipType: 'CO_PURCHASED_WITH',
        relationshipProperties: {{weight: weight}}
    }}
);
"""

G, result = gds.graph.project(
    graph_name="copurchase",
    query=query,
    undirected_relationship_types=[REL_TYPE],
)
print(f"nodes         : {G.node_count():,}")
print(f"relationships : {G.relationship_count():,}")

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 46, Finished, Available, Finished, False)

Graph creation from Triplets:   0%|          | 0.0/100 [00:00<?, ?%/s]

nodes         : 6,932
relationships : 596,442


StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 47, Finished, Available, Finished, False)

StatementMeta(, , -1, Finished, , Finished, True)

RejectSilentExecuteRequest: Livy session has failed. Error code: RejectSilentExecuteRequest. Rejected silent execute_request as there is no active session.

In [32]:
q = f"""
MATCH (o:Order)-[:CONTAINS]->(a:Product), (o)-[:CONTAINS]->(b:Product)
WHERE a.{PRODUCT_KEY} < b.{PRODUCT_KEY}
  AND a.orderCount >= {MIN_PRODUCT_ORDERS}
  AND b.orderCount >= {MIN_PRODUCT_ORDERS}
WITH a, b, count(*) AS w
WHERE w >= {MIN_WEIGHT}
UNWIND [{{p: a, o: 1}}, {{p: b, o: 0}}] AS row
WITH row.p AS p, sum(row.o) AS out_degree, count(*) AS undirected_degree
RETURN p.{PRODUCT_KEY} AS product_id, out_degree, undirected_degree
"""
deg = gds.run_cypher(q)
deg["decile"] = pd.qcut(deg["product_id"], 10, labels=False,
                        duplicates="drop") + 1
print(deg.groupby("decile")[["out_degree", "undirected_degree"]]
         .mean().round(1).to_string())

starved = int(((deg["out_degree"] < DEGREE_CUTOFF) &
               (deg["undirected_degree"] >= DEGREE_CUTOFF)).sum())
eligible = int((deg["undirected_degree"] >= DEGREE_CUTOFF).sum())
print(f"\nproducts eligible undirected : {eligible:,}")
print(f"  starved by direction       : {starved:,} "
      f"({100 * starved / max(eligible, 1):.1f}%)")

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 48, Finished, Available, Finished, False)

        out_degree  undirected_degree
decile                               
1             67.4               70.2
2             73.0               83.7
3             56.0               71.4
4             59.5               86.3
5             61.4              107.3
6             43.8               96.2
7             27.2               76.0
8             20.5               75.7
9             16.3               98.6
10             5.2               95.0

products eligible undirected : 3,353
  starved by direction       : 1,122 (33.5%)


## Step 5 · Node similarity

`relationshipTypes` must name the undirected type. Results are written back to
AuraDB as `SIMILAR_TO` and read out with Cypher — node-property streaming fails
on this session build, and writing lets the database resolve internal ids to
real products.

In [33]:
# Idempotent: clear any SIMILAR_TO from a previous run, batched to stay inside
# the transaction memory limit.
while True:
    gone = int(gds.run_cypher("""
        MATCH ()-[r:SIMILAR_TO]->() WITH r LIMIT 50000
        DELETE r RETURN count(*) AS n
    """)["n"].iloc[0])
    if gone == 0:
        break
    print(f"deleted {gone:,}")
print("SIMILAR_TO cleared")

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 49, Finished, Available, Finished, False)

deleted 26,970
SIMILAR_TO cleared


In [34]:
res = gds.nodeSimilarity.write(
    G,
    relationshipTypes=[REL_TYPE],
    relationshipWeightProperty="weight",
    topK=TOP_K,
    degreeCutoff=DEGREE_CUTOFF,
    similarityCutoff=SIMILARITY_CUTOFF,
    writeRelationshipType="SIMILAR_TO",
    writeProperty="score",
)

in_db = int(gds.run_cypher(
    "MATCH ()-[r:SIMILAR_TO]->() RETURN count(r) AS n")["n"].iloc[0])
print(f"SIMILAR_TO in AuraDB : {in_db:,}")
assert in_db > 0, "nothing written - the session staged to its result store"

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 50, Finished, Available, Finished, False)

 Node Similarity:  21%|##1       | 21.24/100 [00:00<?, ?%/s]

Write-Back (graph: copurchase):   0%|          | 0.0/100 [00:00<?, ?%/s]

SIMILAR_TO in AuraDB : 33,528


StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 51, Finished, Available, Finished, False)

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 52, Finished, Available, Finished, False)

StatementMeta(, , -1, Finished, , Finished, True)

RejectSilentExecuteRequest: Livy session has failed. Error code: RejectSilentExecuteRequest. Rejected silent execute_request as there is no active session.

StatementMeta(, , -1, Finished, , Finished, True)

RejectSilentExecuteRequest: Livy session has failed. Error code: RejectSilentExecuteRequest. Rejected silent execute_request as there is no active session.

In [35]:
# Undirected match plus the id inequality gives one row per pair. topK writes
# a->b without guaranteeing b->a, so matching directed would drop pairs.
sim = gds.run_cypher(f"""
MATCH (a:Product)-[r:SIMILAR_TO]-(b:Product)
WHERE a.{PRODUCT_KEY} < b.{PRODUCT_KEY}
WITH a, b, max(r.score) AS similarity_score
RETURN a.{PRODUCT_KEY}  AS product1_id,
       a.{PRODUCT_NAME} AS product1_name,
       b.{PRODUCT_KEY}  AS product2_id,
       b.{PRODUCT_NAME} AS product2_name,
       similarity_score
ORDER BY similarity_score DESC
""")
print(f"{len(sim):,} unique pairs")
print(sim["similarity_score"].describe().to_string())

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 53, Finished, Available, Finished, False)

28,061 unique pairs
count    28061.000000
mean         0.413578
std          0.100451
min          0.052679
25%          0.344958
50%          0.424262
75%          0.487730
max          0.758765


### Checks

Scores of exactly 1.000 mean identical neighbourhoods, which happens when
`degreeCutoff` is too low. The order-count column shows whether high scores are
concentrated in rarely-bought products — if they are, raise
`MIN_PRODUCT_ORDERS`.

In [36]:
n_one = int((sim["similarity_score"] >= 0.999).sum())
print(f"scores == 1.000 : {n_one}")
assert n_one == 0, (f"{n_one} degenerate pairs - raise DEGREE_CUTOFF above "
                    f"{DEGREE_CUTOFF}")

oc = gds.run_cypher(
    f"MATCH (p:Product) RETURN p.{PRODUCT_KEY} AS pid, p.orderCount AS orders")
o = dict(zip(oc["pid"], oc["orders"]))
top = sim.nlargest(20, "similarity_score").copy()
top["orders1"] = top["product1_id"].map(o)
top["orders2"] = top["product2_id"].map(o)
print()
print(top[["product1_name", "product2_name", "similarity_score",
           "orders1", "orders2"]].to_string(index=False))
lo = min(top["orders1"].min(), top["orders2"].min())
print(f"\nmin order count in the top 20: {lo:,.0f}")

allo = pd.concat([sim["product1_id"].map(o), sim["product2_id"].map(o)])
assert allo.min() >= MIN_PRODUCT_ORDERS, (
    f"min orderCount {allo.min()} < MIN_PRODUCT_ORDERS {MIN_PRODUCT_ORDERS}")

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 54, Finished, Available, Finished, False)

scores == 1.000 : 0

              product1_name                         product2_name  similarity_score  orders1  orders2
       Organic Yellow Onion                        Organic Garlic          0.758765  17118.0  16767.0
         Crispy Rice Cereal         White Unscented Facial Tissue          0.753776    115.0    101.0
Roasted & Salted Pistachios Kit Kat & Reese's Assorted Miniatures          0.743210    171.0    225.0
          Organic Red Onion                      Organic Cilantro          0.736260  10827.0  10719.0
       Organic Yellow Onion                      Organic Zucchini          0.726932  17118.0  16244.0
             Organic Garlic                      Organic Zucchini          0.722653  16767.0  16244.0
          Fresh Cauliflower            Organic Small Bunch Celery          0.716695  10937.0  10474.0
          Organic Red Onion                     Fresh Cauliflower          0.712348  10827.0  10937.0
                    Carrots                     Fresh Caulifl

## Step 6 · Substitutes or complements

A high similarity score means two products keep the same company. It does not say
whether a shopper wants both or only one — and that decides whether the right
action is a cross-sell or a coupon.

Direct co-purchase supplies the missing half. Compare how often the pair actually
appears in the same basket against chance:

`lift = times_together / (ordersA * ordersB / total_orders)`

- **lift > 1** — bought together more than chance. Complement. Cross-sell.
- **lift < 1** — bought together less than chance. Substitute. Coupon.

In [37]:
q = f"""
MATCH (o:Order)-[:CONTAINS]->(p1:Product), (o)-[:CONTAINS]->(p2:Product)
WHERE p1.{PRODUCT_KEY} < p2.{PRODUCT_KEY}
  AND p1.orderCount >= {MIN_PRODUCT_ORDERS}
  AND p2.orderCount >= {MIN_PRODUCT_ORDERS}
WITH p1, p2, count(*) AS w
WHERE w >= {MIN_WEIGHT}
RETURN p1.{PRODUCT_KEY} AS lo, p2.{PRODUCT_KEY} AS hi, w AS times_together
"""
cop = gds.run_cypher(q)
n_orders = int(gds.run_cypher(
    "MATCH (o:Order) RETURN count(o) AS n")["n"].iloc[0])

sim["lo"] = sim[["product1_id", "product2_id"]].min(axis=1)
sim["hi"] = sim[["product1_id", "product2_id"]].max(axis=1)
sim = sim.merge(cop, on=["lo", "hi"], how="left")

o1 = sim["product1_id"].map(o)
o2 = sim["product2_id"].map(o)
sim["lift"] = (sim["times_together"] / (o1 * o2 / n_orders)).round(3)
sim["relationship"] = np.where(sim["lift"] >= 1.0, "complement", "substitute")
sim = sim.drop(columns=["lo", "hi"])

print(sim["relationship"].value_counts().to_string())
print()
for kind in ("complement", "substitute"):
    print(f"--- top {kind}s")
    print(sim[sim["relationship"] == kind]
          .nlargest(8, "similarity_score")
          [["product1_name", "product2_name", "similarity_score",
            "times_together", "lift"]].to_string(index=False))
    print()

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 55, Finished, Available, Finished, False)

relationship
substitute    21060
complement     7001

--- top complements
              product1_name                         product2_name  similarity_score  times_together     lift
       Organic Yellow Onion                        Organic Garlic          0.758765          3380.0    5.832
         Crispy Rice Cereal         White Unscented Facial Tissue          0.753776            31.0 1321.860
Roasted & Salted Pistachios Kit Kat & Reese's Assorted Miniatures          0.743210            21.0  270.323
          Organic Red Onion                      Organic Cilantro          0.736260          1317.0    5.620
       Organic Yellow Onion                      Organic Zucchini          0.726932          1968.0    3.505
             Organic Garlic                      Organic Zucchini          0.722653          1594.0    2.899
          Fresh Cauliflower            Organic Small Bunch Celery          0.716695           808.0    3.493
          Organic Red Onion                     Fresh 

## Step 7 · Best item for a basket

Pairwise scores answer "what is like this product". A basket needs them summed:
score each candidate by its total similarity to everything already in the cart,
excluding what is there.

`n_matches` is the confidence signal — a candidate similar to five basket items
is a safer call than one similar to a single item at a higher score.

In [38]:
def recommend(basket, top=5, kind=None):
    """basket: list of product names. kind: 'complement', 'substitute', None."""
    df = sim if kind is None else sim[sim["relationship"] == kind]
    hit = df[df["product1_name"].isin(basket) | df["product2_name"].isin(basket)]
    if hit.empty:
        return pd.DataFrame()
    out = hit.assign(candidate=np.where(hit["product1_name"].isin(basket),
                                        hit["product2_name"],
                                        hit["product1_name"]))
    out = out[~out["candidate"].isin(basket)]
    return (out.groupby("candidate")
               .agg(basket_affinity=("similarity_score", "sum"),
                    best_match=("similarity_score", "max"),
                    n_matches=("similarity_score", "count"),
                    mean_lift=("lift", "mean"))
               .sort_values("basket_affinity", ascending=False)
               .head(top).round(3))

basket = ["Organic Yellow Onion", "Fresh Cauliflower", "Large Lemon"]
print("basket:", ", ".join(basket))
print()
print("CROSS-SELL (complements - add to the basket)")
print(recommend(basket, kind="complement").to_string())
print()
print("COUPON (substitutes - switch them to these)")
print(recommend(basket, kind="substitute").to_string())

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 56, Finished, Available, Finished, False)

basket: Organic Yellow Onion, Fresh Cauliflower, Large Lemon

CROSS-SELL (complements - add to the basket)
                        basket_affinity  best_match  n_matches  mean_lift
candidate                                                                
Organic Grape Tomatoes            1.814       0.664          3      2.428
Organic Garlic                    1.352       0.759          2      4.372
Limes                             1.335       0.703          2      3.336
Organic Zucchini                  1.316       0.727          2      2.914
Organic Cucumber                  1.281       0.652          2      2.760

COUPON (substitutes - switch them to these)
Empty DataFrame
Columns: []
Index: []


## Step 8 · Results

In [39]:
for name in ("Organic Yellow Onion", "Banana", "Organic Hass Avocado"):
    sub = sim[(sim["product1_name"] == name) | (sim["product2_name"] == name)]
    if sub.empty:
        print(f"\n{name}: no pairs cleared the cutoffs")
        continue
    sub = sub.nlargest(5, "similarity_score").copy()
    sub["other"] = sub.apply(
        lambda r: r["product2_name"] if r["product1_name"] == name
                  else r["product1_name"], axis=1)
    print(f"\n{name}")
    for _, r in sub.iterrows():
        print(f"   {r['similarity_score']:.4f}  {r['other']}")

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 57, Finished, Available, Finished, False)


Organic Yellow Onion
   0.7588  Organic Garlic
   0.7269  Organic Zucchini
   0.6519  Organic Cucumber
   0.6324  Organic Lemon
   0.6320  Limes

Banana
   0.5710  Bag of Organic Bananas
   0.5242  Organic Strawberries
   0.4926  Organic Baby Spinach
   0.4529  Organic Hass Avocado
   0.3705  Organic Avocado

Organic Hass Avocado
   0.7078  Organic Baby Spinach
   0.7060  Organic Strawberries
   0.6097  Bag of Organic Bananas
   0.5703  Organic Raspberries
   0.5695  Organic Avocado


## Step 9 · Write to OneLake

Attach the lakehouse to this notebook first (Explorer → Add lakehouse).

In [40]:
COLS = ["product1_id", "product1_name", "product2_id", "product2_name",
        "similarity_score", "times_together", "lift", "relationship"]
out = sim[[c for c in COLS if c in sim.columns]].copy()
out["product1_id"] = out["product1_id"].astype("int64")
out["product2_id"] = out["product2_id"].astype("int64")
for c in ("times_together", "lift"):
    if c in out.columns:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)
sdf = spark.createDataFrame(out.round({"similarity_score": 6, "lift": 3}))
(sdf.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(OUTPUT_TABLE))
print(f"wrote {OUTPUT_TABLE}: {sdf.count():,} rows")
print(", ".join(sdf.columns))

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 58, Finished, Available, Finished, False)

wrote scored_results: 28,061 rows
product1_id, product1_name, product2_id, product2_name, similarity_score, times_together, lift, relationship


In [41]:
%%sql
WITH targets AS (
    SELECT 'Organic Yellow Onion' AS target UNION ALL
    SELECT 'Large Lemon'                    UNION ALL
    SELECT 'Fresh Cauliflower'
),
pairs AS (
    SELECT t.target,
           CASE WHEN s.product1_name = t.target THEN s.product2_name
                ELSE s.product1_name END AS recommended,
           s.similarity_score
    FROM scored_results s
    JOIN targets t
      ON s.product1_name = t.target OR s.product2_name = t.target
)
SELECT target, recommended, similarity_score
FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY target
                                   ORDER BY similarity_score DESC) AS rn
      FROM pairs) r
WHERE rn <= 3
ORDER BY target, similarity_score DESC

StatementMeta(, 858a50e6-16f2-4fc1-9340-7a798229470d, 59, Finished, Available, Finished, False)

<Spark SQL result set with 9 rows and 3 fields>

## Grounding text for the data agent

> `scored_results` holds pairs of products that are similar because they are
> bought alongside the same other products — second-order co-purchase
> similarity, not direct co-purchase frequency. `similarity_score` is weighted
> Jaccard overlap of each product's co-purchase neighbourhood, from 0 to 1.
> Ubiquitous products such as bananas score low against everything by design.
> Every row is above a similarity floor, so all are usable. A high score means
> two products keep the same company — they may be substitutes or complements,
> and the score does not distinguish those.

## Cleanup

In [ ]:
if gds.graph.exists("copurchase")["exists"]:
    gds.graph.drop("copurchase")

# sessions.delete(session_name=SESSION_NAME)
for s in sessions.list():
    print(s)

## Troubleshooting

| Symptom | Fix |
|---|---|
| `MemoryPoolOutOfMemoryError` | raise `MIN_PRODUCT_ORDERS`; Step 2's table sizes it |
| very few products in the projection | `MIN_PRODUCT_ORDERS` too high — check coverage in Step 2 before projecting |
| mean degree above ~150 | graph too dense; raise `MIN_WEIGHT` rather than the order floor |
| `FlightInternalError ... Cannot map null into type boolean` | node-property streaming is unsupported on this session; the notebook writes `SIMILAR_TO` back to AuraDB instead |
| `bottomK` must be within range [1, ...] | do not pass `bottomK` at all when using `topK` |
| top scores on products with few hundred orders | raise `MIN_PRODUCT_ORDERS` — a handful of near-identical baskets inflates Jaccard |
| pairs scoring exactly 1.000 | raise `DEGREE_CUTOFF` |
| similarity correlates with `productId` | the graph is still directed — check the orientation cell printed `UNDIRECTED` |
| `dataConfig.undirectedRelationshipTypes contained a list of String` | the key belongs on `gds.graph.project()` as `undirected_relationship_types=[...]`, not in the Cypher map |
| assertion that the projection is still directed | `undirected_relationship_types` missing from the `project()` call, or naming a type the projection does not create |
| `relationshipsWritten: 0` from `write` | the session staged to its result store; the assert catches it before the read-back returns an empty frame |
| projection returns 0 nodes | `p.orderCount` is unset — run the popularity cell before projecting, `null >= N` filters everything out |
| `AttributeError` on `AlgorithmCategory.SIMILARITY` | absent on graphdatascience 1.22; handled by the fallback |
| `NameError: name 'productId' is not defined` | Cypher map literals in an f-string need `{{` `}}` |
| `ModuleNotFoundError: graphdatascience` | the `%pip` kernel restart — run the install cell again, alone |


---

## Making this reliable — use a Fabric Environment

`%pip install` is fine for exploring and a liability on stage: it costs ~30
seconds, restarts the kernel, wipes your variables, and is unsupported in High
Concurrency sessions. Libraries installed inline are scoped to the single
session, so every restart pays the cost again.

For anything you intend to demo:

1. **Workspace → New item → Environment.**
2. **Public libraries → Add from PyPI →** `graphdatascience`.
3. **Save**, then **Publish** (publishing takes a few minutes — do it the day
   before, not the morning of).
4. In the notebook, set **Environment** in the toolbar to that environment.
5. Delete or comment out the `%pip install` cell entirely.

The import then just works on a cold session, with no restart and no variable
loss.

**Also worth knowing:** inline `%pip` is disabled by default in *pipeline* runs.
If you schedule this notebook, either use an Environment or pass
`_inlineInstallationEnabled = True` as a boolean parameter on the notebook
activity.

**Sources:** [Manage Apache Spark libraries](https://learn.microsoft.com/en-us/fabric/data-engineering/library-management)
· [Manage libraries in Fabric environments](https://learn.microsoft.com/en-us/fabric/data-engineering/environment-manage-library)
